# 😀 จำแนกอารมณ์จากใบหน้า (Facial Emotion Recognition) — สร้าง/นำเข้าชุดข้อมูลเพื่อส่งเข้าโมเดล Computer Vision

Notebook นี้ทำงานบน **Google Colab** ครอบคลุมขั้นตอนเตรียมชุดข้อมูลจนพร้อมส่งเข้าโมเดล:

| ขั้น | เนื้อหา |
|---|---|
| 1 | ตั้งค่าสภาพแวดล้อม (GPU, ไลบรารี) |
| 2 | **นำเข้าชุดข้อมูล** FER-2013 จาก Kaggle (ภาพใบหน้า 48×48 สีเทา 7 อารมณ์) |
| 3 | **สร้างชุดข้อมูลเอง** — ถ่ายจากเว็บแคมหรืออัปโหลดรูป → ตรวจจับใบหน้า → ครอปเป็น 48×48 |
| 4 | สำรวจข้อมูล (EDA): จำนวนภาพต่อคลาส, ตัวอย่างภาพ, ตรวจไฟล์เสีย/ซ้ำ |
| 5 | Preprocessing: แบ่ง train/val/test, normalize, augmentation, class weight |
| 6 | ส่งข้อมูลเข้าโมเดล CNN (baseline) เพื่อยืนยันว่าชุดข้อมูลใช้งานได้ + ประเมินผล |
| 7 | บันทึกชุดข้อมูล/โมเดลลง Google Drive และทดสอบทำนายภาพใหม่ |

**คลาสอารมณ์ (7 คลาส):** angry (โกรธ), disgust (รังเกียจ), fear (กลัว), happy (มีความสุข), neutral (เฉย ๆ), sad (เศร้า), surprise (ประหลาดใจ)

> 💡 ก่อนเริ่ม: เมนู **Runtime → Change runtime type → T4 GPU** เพื่อให้ train เร็วขึ้น

## 1. ตั้งค่าสภาพแวดล้อม

In [ ]:
import os, shutil, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = 48          # ขนาดภาพมาตรฐานของ FER-2013
BATCH_SIZE = 64
EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
EMOTIONS_TH = {"angry": "โกรธ", "disgust": "รังเกียจ", "fear": "กลัว", "happy": "มีความสุข",
               "neutral": "เฉย ๆ", "sad": "เศร้า", "surprise": "ประหลาดใจ"}

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "ไม่พบ GPU (ใช้ CPU ได้แต่จะช้ากว่า)")

## 2. นำเข้าชุดข้อมูล FER-2013 จาก Kaggle

ใช้ `kagglehub` ดาวน์โหลดชุดข้อมูล [msambare/fer2013](https://www.kaggle.com/datasets/msambare/fer2013) (~35,000 ภาพ) ซึ่งจัดเป็นโฟลเดอร์ `train/<อารมณ์>/*.jpg` และ `test/<อารมณ์>/*.jpg` อยู่แล้ว

> หากดาวน์โหลดไม่ได้เพราะต้องยืนยันตัวตน: ไปที่ Kaggle → Settings → **Create New Token** แล้วรัน `kagglehub.login()` หรือใส่ `KAGGLE_USERNAME` / `KAGGLE_KEY` ในแถบ 🔑 Secrets ของ Colab

In [ ]:
!pip -q install kagglehub
import kagglehub

src_path = kagglehub.dataset_download("msambare/fer2013")
print("ดาวน์โหลดไว้ที่:", src_path)

# คัดลอกมาไว้ที่ /content เพื่อให้แก้ไข/เพิ่มภาพของเราเองได้
DATA_DIR = Path("/content/fer2013")
if not DATA_DIR.exists():
    shutil.copytree(src_path, DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

### (ทางเลือก) นำเข้าชุดข้อมูลจากไฟล์ zip ของตัวเอง
ถ้ากลุ่มมีชุดข้อมูลของตัวเองอยู่แล้ว ให้จัดเป็น `train/<อารมณ์>/...` และ `test/<อารมณ์>/...` แล้ว zip อัปโหลด — ตั้ง `USE_OWN_ZIP = True`

In [ ]:
USE_OWN_ZIP = False

if USE_OWN_ZIP:
    from google.colab import files
    uploaded = files.upload()               # เลือกไฟล์ .zip
    zip_name = next(iter(uploaded))
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    shutil.unpack_archive(zip_name, DATA_DIR)
    print(sorted(os.listdir(DATA_DIR)))

## 3. สร้างชุดข้อมูลเอง (เก็บภาพใบหน้าของสมาชิกกลุ่ม)

ขั้นตอน: **ถ่ายภาพ/อัปโหลดรูป → ตรวจจับใบหน้าด้วย Haar Cascade (OpenCV) → ครอปเฉพาะใบหน้า → แปลงเป็นสีเทา 48×48** ให้รูปแบบเดียวกับ FER-2013 แล้วบันทึกลง `my_faces/<อารมณ์>/`

ภาพที่สร้างเองจะถูกรวมเข้าชุด train ในขั้นถัดไป (ตั้ง `MERGE_MY_FACES = True`)

In [ ]:
MY_DIR = Path("/content/my_faces")
for e in EMOTIONS:
    (MY_DIR / e).mkdir(parents=True, exist_ok=True)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def crop_face(img_bgr, size=IMG_SIZE):
    """ตรวจจับใบหน้าที่ใหญ่ที่สุด ครอป แปลงเป็นสีเทา ย่อเป็น size×size; ไม่พบใบหน้าคืน None"""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    m = int(0.1 * w)  # เผื่อขอบเล็กน้อยให้เห็นคิ้ว/คาง
    x0, y0 = max(x - m, 0), max(y - m, 0)
    face = gray[y0:y + h + m, x0:x + w + m]
    return cv2.resize(face, (size, size), interpolation=cv2.INTER_AREA)

def save_face(face, emotion):
    assert emotion in EMOTIONS, f"emotion ต้องเป็นหนึ่งใน {EMOTIONS}"
    out = MY_DIR / emotion / f"{emotion}_{len(os.listdir(MY_DIR / emotion)):04d}.png"
    cv2.imwrite(str(out), face)
    return out

### 3.1 ถ่ายภาพจากเว็บแคม (ทำท่าทางตามอารมณ์ที่เลือก แล้วกดปุ่ม Capture)

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(quality=0.9):
    js = Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const btn = document.createElement('button');
        btn.textContent = 'Capture';
        div.appendChild(btn);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => btn.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }''')
    display(js)
    data = eval_js(f"takePhoto({quality})")
    buf = np.frombuffer(b64decode(data.split(",")[1]), np.uint8)
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)

def capture(emotion, n=5):
    """ถ่าย n ภาพสำหรับอารมณ์ที่กำหนด"""
    for i in range(n):
        print(f"[{i+1}/{n}] ทำหน้า '{emotion}' ({EMOTIONS_TH[emotion]}) แล้วกด Capture")
        face = crop_face(take_photo())
        if face is None:
            print("  ⚠️ ไม่พบใบหน้า ลองใหม่"); continue
        print("  ✅ บันทึก:", save_face(face, emotion))

# ตัวอย่าง: ถ่ายหน้ายิ้ม 5 ภาพ (เปลี่ยนชื่ออารมณ์แล้วรันซ้ำให้ครบทุกคลาส)
# capture("happy", n=5)

### 3.2 หรืออัปโหลดรูปภาพ (ตั้งชื่อไฟล์ขึ้นต้นด้วยอารมณ์ เช่น `happy_01.jpg`, `sad_bank.png`)

In [ ]:
def import_uploaded_photos():
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        emotion = name.split("_")[0].lower()
        if emotion not in EMOTIONS:
            print(f"ข้าม {name}: ชื่อไฟล์ต้องขึ้นต้นด้วย {EMOTIONS}"); continue
        img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        face = None if img is None else crop_face(img)
        print(name, "→", save_face(face, emotion) if face is not None else "ไม่พบใบหน้า")

# import_uploaded_photos()

In [ ]:
# รวมภาพที่สร้างเองเข้าชุด train
MERGE_MY_FACES = True

my_counts = {e: len(os.listdir(MY_DIR / e)) for e in EMOTIONS}
print("ภาพที่สร้างเอง:", my_counts)
if MERGE_MY_FACES and sum(my_counts.values()) > 0:
    for e in EMOTIONS:
        for f in (MY_DIR / e).iterdir():
            shutil.copy(f, DATA_DIR / "train" / e / f"my_{f.name}")
    print("รวมเข้าชุด train แล้ว")

## 4. สำรวจข้อมูล (EDA)
สร้างตาราง metadata ของทุกภาพ (path, label, split) แล้วตรวจสอบจำนวนภาพต่อคลาส, ไฟล์เสีย และภาพซ้ำ

In [ ]:
rows = []
for split in ["train", "test"]:
    for e in EMOTIONS:
        for f in sorted((DATA_DIR / split / e).iterdir()):
            rows.append({"filepath": str(f), "label": e, "split": split,
                         "source": "own" if f.name.startswith("my_") else "fer2013"})
meta = pd.DataFrame(rows)
print("จำนวนภาพทั้งหมด:", len(meta))
counts = meta.pivot_table(index="label", columns="split", values="filepath", aggfunc="count")
counts["อารมณ์"] = counts.index.map(EMOTIONS_TH)
counts

In [ ]:
# กราฟจำนวนภาพต่อคลาส (ชุด train) — ใช้สีเดียว เพราะเป็นการเปรียบเทียบขนาด
train_counts = meta[meta.split == "train"].label.value_counts().reindex(EMOTIONS)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(train_counts.index, train_counts.values, color="#3b6fd4", width=0.6)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title("Images per class (train) - note the small 'disgust' class")
ax.set_ylabel("images")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# ตัวอย่างภาพ 6 ภาพต่อคลาส
fig, axes = plt.subplots(len(EMOTIONS), 6, figsize=(8, 10))
for r, e in enumerate(EMOTIONS):
    sample = meta[(meta.split == "train") & (meta.label == e)].sample(6, random_state=SEED)
    for c, fp in enumerate(sample.filepath):
        axes[r, c].imshow(cv2.imread(fp, cv2.IMREAD_GRAYSCALE), cmap="gray")
        axes[r, c].axis("off")
    axes[r, 0].set_title(e, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ตรวจคุณภาพข้อมูล: ไฟล์เสีย / ขนาดผิด / ภาพซ้ำ (hash ของพิกเซล)
bad, shapes, hashes = [], {}, []
for fp in meta.filepath:
    img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
    if img is None:
        bad.append(fp); hashes.append(None); continue
    shapes[img.shape] = shapes.get(img.shape, 0) + 1
    hashes.append(hashlib.md5(img.tobytes()).hexdigest())
meta["hash"] = hashes
print("ไฟล์เสีย:", len(bad))
print("ขนาดภาพที่พบ:", shapes)

dup = meta[meta.hash.notna() & meta.duplicated("hash", keep="first")]
print("ภาพซ้ำ:", len(dup))
leak = set(meta[meta.split == "train"].hash) & set(meta[meta.split == "test"].hash)
print("ภาพที่ซ้ำกันระหว่าง train กับ test (data leakage):", len(leak))

In [ ]:
# ลบไฟล์เสียและภาพซ้ำออก (รวมภาพใน train ที่ซ้ำกับ test เพื่อกัน leakage)
REMOVE_DUPLICATES = True
if REMOVE_DUPLICATES:
    to_remove = set(bad) | set(dup.filepath) | set(
        meta[(meta.split == "train") & meta.hash.isin(leak)].filepath)
    for fp in to_remove:
        os.remove(fp)
    meta = meta[~meta.filepath.isin(to_remove)].reset_index(drop=True)
    print("ลบแล้ว", len(to_remove), "ไฟล์ | เหลือ", len(meta), "ภาพ")

meta.drop(columns="hash").to_csv("/content/dataset_metadata.csv", index=False)
print("บันทึก metadata → /content/dataset_metadata.csv")

## 5. Preprocessing และสร้าง `tf.data` pipeline

- แบ่ง **train 85% / validation 15%** จากโฟลเดอร์ train และใช้โฟลเดอร์ test เป็น **test set**
- อ่านเป็นภาพสีเทา 48×48, label แบบ one-hot
- **Normalize** พิกเซล 0–255 → 0–1 (ทำในโมเดลด้วยเลเยอร์ `Rescaling`)
- **Data augmentation** (เฉพาะตอน train): พลิกซ้าย-ขวา, หมุน, ซูม, เลื่อน
- **Class weight** ชดเชยคลาสที่มีภาพน้อย (เช่น disgust)

In [ ]:
common = dict(labels="inferred", label_mode="categorical", class_names=EMOTIONS,
              color_mode="grayscale", image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", validation_split=0.15, subset="training", seed=SEED, shuffle=True, **common)
val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", validation_split=0.15, subset="validation", seed=SEED, shuffle=True, **common)
test_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", shuffle=False, **common)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(2000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

x, y = next(iter(train_ds))
print("batch ภาพ:", x.shape, x.dtype, "ค่าพิกเซล", float(tf.reduce_min(x)), "-", float(tf.reduce_max(x)))
print("batch label:", y.shape)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

y_train_labels = meta[meta.split == "train"].label.map(EMOTIONS.index).values
weights = compute_class_weight("balanced", classes=np.arange(len(EMOTIONS)), y=y_train_labels)
class_weight = dict(enumerate(weights))
pd.DataFrame({"emotion": EMOTIONS, "weight": weights.round(2)})

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

# ดูผล augmentation ของภาพเดียวกัน
img = x[:1]
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    out = img if i == 0 else augment(img, training=True)
    ax.imshow(out[0, ..., 0], cmap="gray"); ax.axis("off")
axes[0].set_title("original", fontsize=9)
plt.show()

## 6. ส่งชุดข้อมูลเข้าโมเดล CNN (Baseline)
โมเดล CNN ขนาดเล็ก 4 บล็อก (Conv–BatchNorm–MaxPool–Dropout) เพื่อยืนยันว่าชุดข้อมูลพร้อมใช้งาน
ความแม่นยำที่คาดหวังบน FER-2013 สำหรับโมเดลลักษณะนี้ประมาณ **60–67%** (มนุษย์ทำได้ราว 65%)

In [ ]:
def build_model(num_classes=len(EMOTIONS)):
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 1))
    x = augment(inputs)
    x = layers.Rescaling(1.0 / 255)(x)
    for filters in [32, 64, 128, 256]:
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="emotion_cnn")

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
EPOCHS = 40
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint("/content/emotion_cnn_best.keras", monitor="val_accuracy", save_best_only=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    class_weight=class_weight, callbacks=callbacks)

In [ ]:
# กราฟการเรียนรู้: train vs validation
h = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(h[metric], color="#3b6fd4", lw=2, label="train")
    ax.plot(h[f"val_{metric}"], color="#e07b28", lw=2, label="validation")
    ax.set_title(metric); ax.set_xlabel("epoch")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.3); ax.legend(frameon=False)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}\n")

y_true = np.concatenate([np.argmax(y, axis=1) for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)
print(classification_report(y_true, y_pred, target_names=EMOTIONS, digits=3))

cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=EMOTIONS).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_title("Confusion matrix (normalized by true class)")
plt.xticks(rotation=45); plt.show()

## 7. บันทึกผลลง Google Drive และทดสอบกับภาพใหม่

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = Path("/content/drive/MyDrive/emotion_project")
OUT.mkdir(parents=True, exist_ok=True)

model.save(OUT / "emotion_cnn.keras")
shutil.copy("/content/dataset_metadata.csv", OUT / "dataset_metadata.csv")
shutil.make_archive(str(OUT / "emotion_dataset"), "zip", DATA_DIR)   # ชุดข้อมูลที่ทำความสะอาดแล้ว
if any(MY_DIR.rglob("*.png")):
    shutil.make_archive(str(OUT / "my_faces"), "zip", MY_DIR)        # ภาพที่กลุ่มสร้างเอง
print("บันทึกไว้ที่", OUT, ":", os.listdir(OUT))

In [ ]:
def predict_emotion(img_bgr):
    face = crop_face(img_bgr)
    if face is None:
        print("ไม่พบใบหน้า"); return
    probs = model.predict(face[None, ..., None].astype("float32"), verbose=0)[0]
    top = int(np.argmax(probs))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={"width_ratios": [1, 2]})
    a1.imshow(face, cmap="gray"); a1.axis("off")
    a1.set_title(f"{EMOTIONS[top]} ({probs[top]:.0%})")
    a2.barh(EMOTIONS, probs, color=["#3b6fd4" if i == top else "#c5cbd6" for i in range(len(EMOTIONS))])
    a2.invert_yaxis(); a2.set_xlim(0, 1); a2.spines[["top", "right"]].set_visible(False)
    plt.show()
    print("ทำนาย:", EMOTIONS[top], f"({EMOTIONS_TH[EMOTIONS[top]]})")

# ทดสอบด้วยเว็บแคม
# predict_emotion(take_photo())

## สรุป
- นำเข้าชุดข้อมูล **FER-2013** (7 อารมณ์, 48×48 สีเทา) และเสริมด้วย **ภาพที่สร้างเอง** ผ่านเว็บแคม/อัปโหลด ซึ่งผ่านการตรวจจับและครอปใบหน้าให้อยู่ในรูปแบบเดียวกัน
- ทำความสะอาดข้อมูล: ลบไฟล์เสีย ภาพซ้ำ และภาพที่รั่วระหว่าง train/test
- เตรียมข้อมูลด้วย `tf.data` + normalization + augmentation + class weight แล้วส่งเข้าโมเดล CNN สำเร็จ
- ข้อสังเกต: คลาส **disgust** มีภาพน้อยมาก และ **fear/sad** มักสับสนกัน — แนวทางพัฒนาต่อ เช่น เก็บภาพเพิ่มในคลาสที่น้อย หรือใช้ transfer learning (เช่น MobileNetV2/EfficientNet กับภาพขยายเป็น 96×96 RGB)